In [1]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [2]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

openai= OpenAI(api_key=api_key)

client = chromadb.PersistentClient(path="./chroma_db")

In [5]:
# OpenAI embedding function
embedding_function = OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)

In [6]:
# Create collection
collection = client.get_or_create_collection(
    name="netflix_titles",
    embedding_function=embedding_function
)

In [8]:
# Load data
with open("netflix_titles.json", "r", encoding="utf-8") as f:
    data = json.load(f)

documents = []
metadatas = []
ids = []

for item in data:
    document = f"""
    Title: {item['title']}
    Type: {item['type']}
    Genres: {item['listed_in']}
    Description: {item['description']}
    """

    documents.append(document)

    metadatas.append({
        "title": item["title"],
        "type": item["type"],
        "release_year": item["release_year"]
    })

    ids.append(item["show_id"])

In [9]:
print(f"Adding {len(documents)} documents to the collection.")
print(documents[0])

Adding 5 documents to the collection.

    Title: Dog Days
    Type: Movie
    Genres: Comedy, Drama
    Description: A group of people discover how their lives become connected through their lovable dogs.
    


In [10]:
# Insert data
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)


In [11]:
print(f"Inserted {len(ids)} records")

Inserted 5 records


In [12]:
result = collection.query(
    query_texts=["films about dogs"],
    n_results=3
)

In [14]:
print(result)

{'ids': [['s1', 's2', 's3']], 'embeddings': None, 'documents': [['\n    Title: Dog Days\n    Type: Movie\n    Genres: Comedy, Drama\n    Description: A group of people discover how their lives become connected through their lovable dogs.\n    ', '\n    Title: Benji\n    Type: Movie\n    Genres: Family Movies\n    Description: A determined dog helps rescue two kidnapped children.\n    ', '\n    Title: The Secret Life of Pets\n    Type: Movie\n    Genres: Animation, Family\n    Description: A terrier and his canine friends embark on an adventure through New York City.\n    ']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'release_year': 2018, 'type': 'Movie', 'title': 'Dog Days'}, {'type': 'Movie', 'release_year': 2018, 'title': 'Benji'}, {'type': 'Movie', 'release_year': 2016, 'title': 'The Secret Life of Pets'}]], 'distances': [[0.3917490243911743, 0.5087102651596069, 0.5133143663406372]]}


In [15]:
print("\nTop Matches:\n")

for i in range(len(result["ids"][0])):
    print("=" * 50)
    print("ID:", result["ids"][0][i])
    print("Metadata:", result["metadatas"][0][i])
    print("Document:", result["documents"][0][i])
    print("Distance:", result["distances"][0][i])


Top Matches:

ID: s1
Metadata: {'release_year': 2018, 'type': 'Movie', 'title': 'Dog Days'}
Document: 
    Title: Dog Days
    Type: Movie
    Genres: Comedy, Drama
    Description: A group of people discover how their lives become connected through their lovable dogs.
    
Distance: 0.3917490243911743
ID: s2
Metadata: {'type': 'Movie', 'release_year': 2018, 'title': 'Benji'}
Document: 
    Title: Benji
    Type: Movie
    Genres: Family Movies
    Description: A determined dog helps rescue two kidnapped children.
    
Distance: 0.5087102651596069
ID: s3
Metadata: {'type': 'Movie', 'release_year': 2016, 'title': 'The Secret Life of Pets'}
Document: 
    Title: The Secret Life of Pets
    Type: Movie
    Genres: Animation, Family
    Description: A terrier and his canine friends embark on an adventure through New York City.
    
Distance: 0.5133143663406372
